<a href="https://colab.research.google.com/github/muhammad-usman-tech/cafe-sales-data-cleaning/blob/main/Cafe_Sales_ETL_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
df=pd.read_csv('/content/dirty_cafe_sales.csv')
# little bit of exploration and inspecting
print(df.head(7))
# df.info()
df.shape #we got 10 thousand records and the 8 attributes (10000, 8)
#while inspecting i saw some values like ERROR , UNKNOWN

# starting with the little daa clening and discovering NaN's
obj=df.select_dtypes(include='object').columns
obj
for ob in obj:
  df[ob]=df[ob].str.strip()
df=df.replace(['ERROR','UNKNOWN',''],np.nan) #in this way all the nan will be more viewable

# now working with the columns names
df.columns=(df.columns
            .str.lower()
            .str.replace(r'[^a-z0-9]+','_',regex=True)
            .str.strip('_'))

# now go to change the data types if needed(lets have a again look on info() or dtypes)
df.info()
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce')
df['price_per_unit']=pd.to_numeric(df['price_per_unit'],errors='coerce')
df['total_spent']=pd.to_numeric(df['total_spent'],errors='coerce')
df['transaction_date']=pd.to_datetime(df['transaction_date'],errors='coerce')

# now move towards to deal with the duplicates
df.duplicated().sum()
df=df.drop_duplicates()
df=df.dropna(subset=['transaction_id'])
df['transaction_id'].duplicated().sum() #got 0 duplicates
# ok so starting with the null checking
null_df=df.isnull().sum()
percent_null=(null_df/len(df))*100
drop_col=percent_null[percent_null>50].index

if not drop_col.empty:
    df = df.drop(columns=drop_col)
else:
    print("No columns found with > 50% null values. Skipping drop.")

# ok now go with the other things obj_columns
null_df=df.isnull().sum()
percent_null=(null_df/len(df))*100
percent_null

# paymemt and location are more then 31 per and they arer the obj data so better to go with the filling with unknown
obj_columns=df.select_dtypes(include='object').columns
df[obj_columns]=df[obj_columns].fillna('UNKNOWN')

# now all the numeric values are the 5 or 4 percent better to fill with the median()
num_col=df.select_dtypes(include='number').columns
df[num_col]=df[num_col].fillna(df[num_col].median())

# now checking
null_df=df.isnull().sum()
percent_null=(null_df/len(df))*100
percent_null
# leaving date column as it is because it can change the trend

# ok now just checking the possible andimpossible values
df['transaction_date'].max() #Timestamp('2023-12-31 00:00:00')
# max date is 31 dec 2023
df['transaction_date'].min()
# min is jan 1st 2023 means it a complete year data
df.info()
# i am watching the quantity is in float and not in int lets see it
df['quantity']=df['quantity'].astype('int')
# as we have dne with the errors in the data type conversion so its ok to convert it herer in the int

df.info()

# some validation and its done
df.isnull().sum()
df.duplicated().sum()
# yes every thing if fine except the date only has 4.6 percent NaN
df.to_csv('cleaned_data.csv',index=True)


  Transaction ID      Item Quantity Price Per Unit Total Spent  \
0    TXN_1961373    Coffee        2            2.0         4.0   
1    TXN_4977031      Cake        4            3.0        12.0   
2    TXN_4271903    Cookie        4            1.0       ERROR   
3    TXN_7034554     Salad        2            5.0        10.0   
4    TXN_3160411    Coffee        2            2.0         4.0   
5    TXN_2602893  Smoothie        5            4.0        20.0   
6    TXN_4433211   UNKNOWN        3            3.0         9.0   

   Payment Method  Location Transaction Date  
0     Credit Card  Takeaway       2023-09-08  
1            Cash  In-store       2023-05-16  
2     Credit Card  In-store       2023-07-19  
3         UNKNOWN   UNKNOWN       2023-04-27  
4  Digital Wallet  In-store       2023-06-11  
5     Credit Card       NaN       2023-03-31  
6           ERROR  Takeaway       2023-10-06  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 